In [1]:
# !pip install getml pyarrow relbench

In [2]:
import getml
from relbench.datasets import get_dataset
from relbench.tasks import get_task

dataset = get_dataset("rel-hm", download=True)
task = get_task("rel-hm", "item-sales", download=True)

getml.engine.launch(in_memory=False)

getml.set_project("hm-item-final")

getML Engine is already running.


Output()

Connected to project 'hm-item-final'.

In [3]:
def load_df(path: str, name: str, roles: getml.data.Roles) -> getml.data.DataFrame:
    """
    Load a DataFrame from a parquet file and save it in getml native format to
    disk if it does not exist yet.
    """
    df = getml.data.DataFrame.from_parquet(
        path,
        name=name,
        roles=roles,
    )
    df.save()
    return df


population_roles = getml.data.Roles(
    join_key=["article_id"],
    target=["sales"],
    time_stamp=["timestamp"],
)

subsets = ("train", "test", "val")
populations = {}
for subset in subsets:
    populations[subset] = load_df(
        f"{dataset.cache_dir}/tasks/item-sales/{subset}.parquet",
        subset,
        population_roles,
    )
    populations[subset].save()

customer_roles = getml.data.Roles(
    join_key=["customer_id"],
    numerical=["age"],
)

customer = load_df(
    f"{dataset.cache_dir}/db/customer.parquet", "customer", customer_roles
)

transaction_roles = getml.data.Roles(
    join_key=["article_id", "customer_id"],
    time_stamp=["t_dat"],
    numerical=["price"],
    categorical=["sales_channel_id"],
)

transaction = load_df(
    f"{dataset.cache_dir}/db/transactions.parquet", "transaction", transaction_roles
)

transaction.set_subroles(["t_dat"], getml.data.subroles.exclude.seasonal)

transaction.save()

article_roles = getml.data.Roles(
    join_key=["article_id"],
    numerical=[
        "department_no",
        "section_no",
        # "perceived_colour_master_id"
    ],
    # categorical=[
    #     "product_type_name",
    #     "graphical_appearance_name",
    #     "colour_group_name",
    #     "perceived_colour_value_name",
    #     "perceived_colour_master_name",
    #     "section_no",
    #     "index_name",
    #     "index_group_name",
    #     "product_group_name",
    #     "department_name",
    # ],
)

    # article.department_no,
    # article.section_no,
    # article.perceived_colour_master_id,

article = load_df(f"{dataset.cache_dir}/db/article.parquet", "article", article_roles)



In [4]:
populations["train"]

name,timestamp,article_id,sales
role,time_stamp,join_key,target
unit,time stamp,,
0,2020-06-01,83325,2.728
1,2020-06-01,50838,0.7581
2,2020-06-01,86874,1.3376
3,2020-06-01,89952,1.3062
4,2020-06-01,79129,0.2711
,...,...,...
5488179,2019-09-09,100865,0
5488180,2019-09-09,101443,0


In [5]:
dataset.test_timestamp.to_datetime64()

numpy.datetime64('2020-09-14T00:00:00')

In [6]:
dm = getml.data.DataModel(population=populations["train"].to_placeholder())
dm.add(getml.data.to_placeholder(customer, transaction, article))
# dm.add(getml.data.to_placeholder(transaction,transaction))

dm.population.join(
    dm.article, on="article_id", relationship=getml.data.relationship.many_to_one
)
dm.population.join(
    dm.transaction, on="article_id", time_stamps=("timestamp", "t_dat"),
    memory=getml.data.time.weeks(1)
)
dm.population.join(
    dm.transaction, on="article_id", time_stamps=("timestamp", "t_dat"),
    # horizon=getml.data.time.weeks(2),
    memory=getml.data.time.weeks(6)
)
dm.transaction.join(
    dm.customer, on="customer_id", relationship=getml.data.relationship.many_to_one
)

container = getml.data.Container(**populations)
container.add(customer, transaction, article)

dm

,data frames,staging table
0,"train, article",TRAIN__STAGING_TABLE_1
1,"transaction, customer",TRANSACTION__STAGING_TABLE_2


In [7]:
container

population
    subset   name       rows   type     
0   train    train   5488184   DataFrame
1   test     test     105542   DataFrame
2   val      val      105542   DataFrame

peripheral
    name              rows   type     
0   customer       1371980   DataFrame
1   transaction   15453651   DataFrame
2   article         105542   DataFrame

In [13]:
pipe = getml.Pipeline(
    tags=["task: item-sales; model:fastprop, seasonal, mapping"],
    data_model=dm,
    preprocessors=[getml.preprocessors.Seasonal(), getml.preprocessors.Mapping()],
    feature_learners=[
        getml.feature_learning.FastProp(
            num_threads=24,
            n_most_frequent=2,
            num_features=200,
            aggregation=getml.feature_learning.FastProp.agg_sets.default
            | {
                # getml.feature_learning.aggregations.COUNT_DISTINCT_OVER_COUNT,
                getml.feature_learning.aggregations.EWMA_1D,
                getml.feature_learning.aggregations.EWMA_7D,
                getml.feature_learning.aggregations.EWMA_30D,
                # getml.feature_learning.aggregations.EWMA_90D,
                # getml.feature_learning.aggregations.EWMA_365D,
                # getml.feature_learning.aggregations.EWMA_TREND_7D,
                # getml.feature_learning.aggregations.EWMA_TREND_30D,
                # getml.feature_learning.aggregations.EWMA_TREND_90D,
                # getml.feature_learning.aggregations.EWMA_TREND_365D,
                getml.feature_learning.aggregations.Q_1,
                getml.feature_learning.aggregations.Q_5,
                getml.feature_learning.aggregations.Q_10,
                getml.feature_learning.aggregations.Q_25,
                getml.feature_learning.aggregations.TIME_SINCE_FIRST_MINIMUM,
                getml.feature_learning.aggregations.TIME_SINCE_LAST_MINIMUM,
                getml.feature_learning.aggregations.TIME_SINCE_LAST_MAXIMUM,
                getml.feature_learning.aggregations.TIME_SINCE_FIRST_MAXIMUM,
            },
        )
    ],
    # include_categorical=True,
    predictors=[getml.predictors.XGBoostRegressor(n_jobs=24)],
    loss_function=getml.feature_learning.loss_functions.SquareLoss,
)

In [9]:
pipe.check(container.train)

Checking data model...

Output()

The pipeline check generated 2 issues labeled INFO and 0 issues labeled WARNING.

,type,label,message
0,INFO,FOREIGN KEYS NOT FOUND,"When joining TRAIN__STAGING_TABLE_1 and TRANSACTION__STAGING_TABLE_2 over 'article_id' and 'article_id', there are no corresponding entries for 31.637642% of entries in 'article_id' in 'TRAIN__STAGING_TABLE_1'. You might want to double-check your join keys."
1,INFO,FOREIGN KEYS NOT FOUND,"When joining TRAIN__STAGING_TABLE_1 and TRANSACTION__STAGING_TABLE_3 over 'article_id' and 'article_id', there are no corresponding entries for 31.637642% of entries in 'article_id' in 'TRAIN__STAGING_TABLE_1'. You might want to double-check your join keys."


In [10]:
pipe.fit(container.train, check = False)

Output()

Trained pipeline.

Time taken: 0:28:20.296057.



Pipeline(data_model='train',
         feature_learners=['FastProp'],
         feature_selectors=[],
         include_categorical=False,
         loss_function='SquareLoss',
         peripheral=['article', 'customer', 'transaction'],
         predictors=['XGBoostRegressor'],
         preprocessors=['Seasonal', 'Mapping'],
         share_selected_features=0.5,
         tags=['task: item-sales; model:fastprop, seasonal, mapping',
               'container-Kzm1W2'])

In [11]:
pipe.score(container.val)
pipe.score(container.test)
pipe.scores

Output()

Output()

,date time,set used,target,mae,rmse,rsquared
0,2024-12-18 19:27:13,train,sales,0.04325,0.2743,0.6941
1,2024-12-18 19:27:44,val,sales,0.05019,0.3873,0.6587
2,2024-12-18 19:28:14,test,sales,0.04544,0.3144,0.7239
